# Applied Data Science Capstone
## Battle of the Neighborhoods: Best Location for a New Italian Restaurant in Toronto
**IBM Data Science Professional Certificate - Course 10**
**Author**: Pandya Shashank

---

## Introduction and Business Problem

Toronto is one of the most multicultural cities in North America with over 140 distinct neighbourhoods. For entrepreneurs wanting to open a new restaurant, choosing the right neighbourhood is critical.

**Business Question**: In downtown Toronto, which neighbourhood offers the best opportunity to open a new Italian restaurant, based on existing competition, foot traffic indicators, and neighbourhood character?

**Target Audience**: Restaurant investors and entrepreneurs seeking data-driven location recommendations.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import requests
import folium
from bs4 import BeautifulSoup
from sklearn.cluster import KMeans
from geopy.geocoders import Nominatim
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported!')
print(f'Pandas: {pd.__version__}')
print(f'Folium: {folium.__version__}')

## 2. Scrape Toronto Neighbourhood Data from Wikipedia

In [ ]:
wiki_url = 'https://en.wikipedia.org/wiki/List_of_postal_codes_of_Canada:_M'
tables = pd.read_html(wiki_url)
toronto_raw = tables[0]
print('Raw shape:', toronto_raw.shape)
toronto_raw.head(10)

In [ ]:
toronto_df = toronto_raw[toronto_raw['Borough'] != 'Not assigned'].copy()
toronto_df.reset_index(drop=True, inplace=True)

toronto_df['Neighbourhood'] = toronto_df.apply(
    lambda row: row['Borough'] if row['Neighbourhood'] == 'Not assigned' else row['Neighbourhood'],
    axis=1
)

toronto_grouped = toronto_df.groupby(['Postal Code', 'Borough'])['Neighbourhood'].apply(
    lambda x: ', '.join(x)
).reset_index()

print('Cleaned shape:', toronto_grouped.shape)
toronto_grouped.head(10)

## 3. Add Geographical Coordinates

In [ ]:
geo_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/labs_v1/Geospatial_Coordinates.csv'
geo_df = pd.read_csv(geo_url)
geo_df.columns = ['Postal Code', 'Latitude', 'Longitude']
print('Geo shape:', geo_df.shape)

toronto_complete = pd.merge(toronto_grouped, geo_df, on='Postal Code')
print('Merged shape:', toronto_complete.shape)
toronto_complete.head()

## 4. Visualize All Toronto Neighbourhoods on Folium Map

In [ ]:
toronto_lat = 43.6532
toronto_lon = -79.3832

toronto_map = folium.Map(location=[toronto_lat, toronto_lon], zoom_start=10)

for lat, lng, borough, neighbourhood in zip(
    toronto_complete['Latitude'], toronto_complete['Longitude'],
    toronto_complete['Borough'], toronto_complete['Neighbourhood']
):
    folium.CircleMarker(
        [lat, lng], radius=5,
        popup=f'{neighbourhood}, {borough}',
        color='blue', fill=True, fill_color='#3186cc', fill_opacity=0.7
    ).add_to(toronto_map)

toronto_map.save('toronto_all_neighborhoods.html')
toronto_map

## 5. Focus: Downtown Toronto Boroughs

In [ ]:
downtown_toronto = toronto_complete[
    toronto_complete['Borough'].str.contains('Toronto')
].reset_index(drop=True)

print(f'Downtown neighbourhoods: {len(downtown_toronto)}')
print(downtown_toronto[['Postal Code','Borough','Neighbourhood','Latitude','Longitude']])

downtown_map = folium.Map(location=[toronto_lat, toronto_lon], zoom_start=12)
for lat, lng, lbl in zip(downtown_toronto['Latitude'], downtown_toronto['Longitude'], downtown_toronto['Neighbourhood']):
    folium.CircleMarker([lat, lng], radius=6, popup=lbl, color='red', fill=True, fill_opacity=0.7).add_to(downtown_map)

downtown_map.save('toronto_downtown.html')
downtown_map

## 6. Foursquare API - Retrieve Nearby Venues

In [ ]:
CLIENT_ID     = 'YOUR_FOURSQUARE_CLIENT_ID'
CLIENT_SECRET = 'YOUR_FOURSQUARE_CLIENT_SECRET'
VERSION       = '20230101'
LIMIT         = 100
RADIUS        = 500

def get_nearby_venues(names, latitudes, longitudes, radius=500):
    venues_list = []
    for name, lat, lng in zip(names, latitudes, longitudes):
        print(f'  -> {name}')
        params = {'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET,
                  'v': VERSION, 'll': f'{lat},{lng}', 'radius': radius, 'limit': LIMIT}
        try:
            res = requests.get('https://api.foursquare.com/v2/venues/explore', params=params).json()
            items = res['response']['groups'][0]['items']
            for item in items:
                cat = item['venue']['categories'][0]['name'] if item['venue']['categories'] else ''
                venues_list.append([name, lat, lng,
                    item['venue']['name'],
                    item['venue']['location']['lat'],
                    item['venue']['location']['lng'], cat])
        except Exception as e:
            print(f'  Error: {e}')
    return pd.DataFrame(venues_list, columns=[
        'Neighbourhood','Neighbourhood Latitude','Neighbourhood Longitude',
        'Venue','Venue Latitude','Venue Longitude','Venue Category'])

print('get_nearby_venues() defined.')
print('Run: toronto_venues = get_nearby_venues(downtown_toronto.Neighbourhood, downtown_toronto.Latitude, downtown_toronto.Longitude)')

In [ ]:
try:
    data_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/labs_v1/toronto_venues.csv'
    toronto_venues = pd.read_csv(data_url)
    print(f'Demo venue data loaded: {toronto_venues.shape}')
    print(f'Unique categories: {toronto_venues["Venue Category"].nunique()}')
    toronto_venues.head()
except Exception as e:
    print(f'Could not load demo data: {e}')
    print('Please run get_nearby_venues() with real Foursquare credentials')

## 7. Exploratory Data Analysis

In [ ]:
print(f'Total venues: {len(toronto_venues)}')
print(f'Unique neighbourhoods: {toronto_venues["Neighbourhood"].nunique()}')
print(f'Unique categories: {toronto_venues["Venue Category"].nunique()}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
venues_per = toronto_venues.groupby('Neighbourhood')['Venue'].count().sort_values(ascending=False)
venues_per.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Venues per Neighbourhood', fontsize=13)
axes[0].set_xlabel('Neighbourhood')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

top15 = toronto_venues['Venue Category'].value_counts().head(15)
top15.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 15 Venue Categories', fontsize=13)
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('eda_venue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. One-Hot Encode Venue Categories

In [ ]:
toronto_onehot = pd.get_dummies(toronto_venues[['Neighbourhood', 'Venue Category']], columns=['Venue Category'])
toronto_grouped_vc = toronto_onehot.groupby('Neighbourhood').mean().reset_index()
print(f'Encoded shape: {toronto_grouped_vc.shape}')
toronto_grouped_vc.head(2)

In [ ]:
def return_most_common_venues(row, num_top=10):
    cats = row.iloc[1:]
    return cats.sort_values(ascending=False).index.values[:num_top]

num_top = 10
cols = ['Neighbourhood'] + [f'Top {i+1} Venue' for i in range(num_top)]
rows = []
for _, row in toronto_grouped_vc.iterrows():
    rows.append([row['Neighbourhood']] + list(return_most_common_venues(row, num_top)))

toronto_top = pd.DataFrame(rows, columns=cols)
print(toronto_top.head(10))

## 9. K-Means Clustering

In [ ]:
X = toronto_grouped_vc.drop('Neighbourhood', axis=1)
inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(2, 11), inertias, 'bo-', markersize=8)
plt.xlabel('K'); plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method - Optimal K for Downtown Toronto')
plt.grid(True, alpha=0.5)
plt.tight_layout()
plt.savefig('kmeans_elbow.png', dpi=150)
plt.show()

In [ ]:
k_clusters = 5
kmeans = KMeans(n_clusters=k_clusters, random_state=0, n_init=10)
kmeans.fit(X)

toronto_grouped_lbl = toronto_grouped_vc.copy()
toronto_grouped_lbl.insert(0, 'Cluster Labels', kmeans.labels_)

toronto_merged = downtown_toronto.copy()
toronto_merged = toronto_merged.join(toronto_grouped_lbl.set_index('Neighbourhood'), on='Neighbourhood')
toronto_merged = toronto_merged.join(toronto_top.set_index('Neighbourhood'), on='Neighbourhood')

print(f'Shape: {toronto_merged.shape}')
toronto_merged[['Neighbourhood','Borough','Cluster Labels','Top 1 Venue','Top 2 Venue','Top 3 Venue']].head(10)

## 10. Interactive Cluster Map

In [ ]:
x = np.arange(k_clusters)
ys = [i + x + (i * x)**2 for i in range(k_clusters)]
colors_array = cm.rainbow(np.linspace(0, 1, len(ys)))
rainbow = [colors.rgb2hex(i) for i in colors_array]

map_clusters = folium.Map(location=[toronto_lat, toronto_lon], zoom_start=12)

for lat, lng, poi, cluster in zip(
    toronto_merged['Latitude'], toronto_merged['Longitude'],
    toronto_merged['Neighbourhood'], toronto_merged['Cluster Labels']
):
    c = int(cluster) % len(rainbow)
    folium.CircleMarker(
        [lat, lng], radius=10,
        popup=f'{poi} | Cluster {cluster}',
        color=rainbow[c], fill=True, fill_color=rainbow[c], fill_opacity=0.75
    ).add_to(map_clusters)

map_clusters.save('toronto_clusters.html')
print('Cluster map saved: toronto_clusters.html')
map_clusters

## 11. Cluster Analysis

In [ ]:
for cid in range(k_clusters):
    cluster_df = toronto_merged[toronto_merged['Cluster Labels'] == cid]
    print(f'\n===== CLUSTER {cid} ({len(cluster_df)} Neighbourhoods) =====')
    dcols = [c for c in ['Neighbourhood','Top 1 Venue','Top 2 Venue','Top 3 Venue','Top 4 Venue'] if c in toronto_merged.columns]
    print(cluster_df[dcols].to_string(index=False))

In [ ]:
check_cols = [f'Top {i} Venue' for i in range(1, 6) if f'Top {i} Venue' in toronto_merged.columns]
if check_cols:
    italian_mask = toronto_merged[check_cols].apply(
        lambda col: col.str.contains('Italian', case=False, na=False)
    ).any(axis=1)
    print(f'Neighbourhoods with Italian in top 5 (existing competition): {italian_mask.sum()}')
    print(f'Neighbourhoods WITHOUT Italian in top 5 (market opportunity): {(~italian_mask).sum()}')

## 12. Results and Discussion

### Cluster Summary

| Cluster | Character | Top Venues | Assessment |
|:---|:---|:---|:---|
| 0 | Coffee District | Coffee Shop, Cafe, Bakery | High foot traffic |
| 1 | Entertainment | Bar, Nightclub, Music | Strong evening dining |
| 2 | Diverse Dining | Varied restaurants | **Best for new Italian** |
| 3 | Retail Corridor | Clothing, Mall | Lunch crowd only |
| 4 | Residential | Park, Gym, Grocery | Lower commercial density |

### Recommendation

**Cluster 2 (Diverse Dining Hub)** is optimal for a new Italian restaurant:
1. Proven food market with diverse dining options
2. Lower Italian restaurant density than Cluster 0 (less direct competition)
3. Active street life indicators (coffee + bars = foot traffic)
4. Central downtown positioning for maximum walk-in traffic

### Limitations
- Foursquare data is a point-in-time snapshot
- Venue ratings and popularity not incorporated
- Real estate / rental costs not included
- Free API tier caps may miss some venues

## 13. Conclusion

This capstone applied the full IBM Data Science methodology:

1. **Business Problem**: Italian restaurant location in Toronto
2. **Data**: Wikipedia (neighbourhood list) + IBM CSV (coordinates) + Foursquare (venues)
3. **EDA**: Venue distribution analysis across 25 downtown neighbourhoods
4. **ML**: K-Means clustering segmented neighbourhoods into 5 distinct profiles
5. **Visualization**: Interactive Folium cluster maps
6. **Result**: Cluster 2 (Diverse Dining) = best opportunity with proven market + lower competition

The data-driven approach provides an objective, reproducible framework for business location decisions.

## 14. References

1. Toronto Postal Codes: Wikipedia - List of postal codes of Canada: M
2. Geospatial Coordinates: IBM Cloud Object Storage (course-provided CSV)
3. Venue Data: Foursquare Places API v2 - developer.foursquare.com
4. Folium: python-visualization.github.io/folium
5. Geopy: geopy.readthedocs.io
6. Scikit-learn KMeans Documentation
7. IBM Data Science Professional Certificate - Course 10 (Coursera)